# Fine-tune MobileFaceNet on African Faces (RFW)

This notebook fine-tunes a pre-trained MobileFaceNet model on the **RFW African subset** to improve face verification accuracy for Nigerian field workers.

**Requirements**: T4 GPU runtime (Runtime > Change runtime type > T4 GPU)

**Output**: `mobilefacenet_african_v2.tflite` — drop-in replacement for the Flutter app

| Spec | Value |
|------|-------|
| Input | `[1, 112, 112, 3]` Float32, normalized `[-1, 1]` |
| Output | `[1, 192]` Float32, L2-normalized |
| Target | 90%+ accuracy on African face pairs |

## Cell 1 — Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q insightface onnx2torch onnxruntime onnx-tf tensorflow \
    torch torchvision scikit-learn matplotlib tqdm gdown huggingface_hub

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Mount Google Drive for saving checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/mobilefacenet_african'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Checkpoints will be saved to: {SAVE_DIR}')
except ImportError:
    SAVE_DIR = './checkpoints'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'Not running in Colab. Saving to: {SAVE_DIR}')

## Cell 2 — Download Pre-trained MobileFaceNet

In [ ]:
import onnx
import onnx2torch
import onnxruntime as ort
import shutil

# ============================================================
# Download MobileFaceNet ONNX from HuggingFace (no GDrive issues)
# ============================================================
ONNX_PATH = '/content/mobilefacenet.onnx'

if not os.path.exists(ONNX_PATH):
    print('Downloading MobileFaceNet ONNX model...\n')

    # Strategy 1: HuggingFace Hub (most reliable, no Google Drive)
    downloaded = False
    try:
        from huggingface_hub import hf_hub_download
        print('Trying HuggingFace Hub (deepinsight/buffalo_sc)...')
        local_path = hf_hub_download(
            repo_id='deepinsight/buffalo_sc',
            filename='w600k_mbf.onnx',
        )
        shutil.copy(local_path, ONNX_PATH)
        print(f'  Downloaded from HuggingFace: {local_path}')
        downloaded = True
    except Exception as e:
        print(f'  HuggingFace failed: {e}')

    # Strategy 2: Direct wget from InsightFace GitHub release assets
    if not downloaded:
        try:
            print('\nTrying direct download via wget...')
            import subprocess
            # buffalo_sc pack from InsightFace releases
            result = subprocess.run([
                'wget', '-q', '--show-progress',
                'https://huggingface.co/deepinsight/buffalo_sc/resolve/main/w600k_mbf.onnx',
                '-O', ONNX_PATH
            ], capture_output=False, timeout=120)
            if result.returncode == 0 and os.path.exists(ONNX_PATH) and os.path.getsize(ONNX_PATH) > 1_000_000:
                downloaded = True
                print('  Downloaded via wget')
        except Exception as e:
            print(f'  wget failed: {e}')

    # Strategy 3: insightface package (uses GDrive internally, may fail)
    if not downloaded:
        try:
            print('\nTrying insightface package model zoo...')
            import insightface
            from insightface.app import FaceAnalysis
            app = FaceAnalysis(name='buffalo_sc', providers=['CPUExecutionProvider'])
            app.prepare(ctx_id=-1, det_size=(112, 112))
            model_dir = os.path.expanduser('~/.insightface/models/buffalo_sc')
            for f in sorted(os.listdir(model_dir)):
                if not f.endswith('.onnx'):
                    continue
                full_path = os.path.join(model_dir, f)
                m = onnx.load(full_path)
                inp = m.graph.input[0].type.tensor_type.shape
                dims = [d.dim_value for d in inp.dim]
                if len(dims) == 4 and dims[2] == 112 and dims[3] == 112:
                    shutil.copy(full_path, ONNX_PATH)
                    print(f'  Found recognition model: {f}')
                    downloaded = True
                    break
        except Exception as e:
            print(f'  insightface failed: {e}')

    if not downloaded:
        raise RuntimeError(
            'All download methods failed.\n'
            'Please manually download w600k_mbf.onnx from:\n'
            '  https://huggingface.co/deepinsight/buffalo_sc/tree/main\n'
            'Upload it to Colab and copy to /content/mobilefacenet.onnx'
        )

print(f'\nONNX model: {ONNX_PATH}')
print(f'Size: {os.path.getsize(ONNX_PATH) / 1024 / 1024:.1f} MB')

# ============================================================
# Inspect the ONNX model
# ============================================================
onnx_model = onnx.load(ONNX_PATH)
input_info = onnx_model.graph.input[0]
output_info = onnx_model.graph.output[0]
input_shape = [d.dim_value for d in input_info.type.tensor_type.shape.dim]
output_shape = [d.dim_value for d in output_info.type.tensor_type.shape.dim]
print(f'Input:  name={input_info.name}, shape={input_shape}')
print(f'Output: name={output_info.name}, shape={output_shape}')

ONNX_EMBEDDING_DIM = output_shape[1] if len(output_shape) == 2 else output_shape[-1]
TARGET_EMBEDDING_DIM = 192  # Must match Flutter app
print(f'\nONNX embedding dim: {ONNX_EMBEDDING_DIM}')
print(f'Flutter target dim: {TARGET_EMBEDDING_DIM}')

# ============================================================
# Verify with ONNX Runtime first
# ============================================================
sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1, 3, 112, 112).astype(np.float32)
onnx_output = sess.run(None, {input_info.name: dummy_np})[0]
print(f'\nONNX Runtime test: input (1,3,112,112) -> output {onnx_output.shape}')

# ============================================================
# Convert ONNX to PyTorch
# ============================================================
print('\nConverting ONNX -> PyTorch...')
pytorch_model_raw = onnx2torch.convert(ONNX_PATH)
pytorch_model_raw = pytorch_model_raw.to(DEVICE)
pytorch_model_raw.eval()


class BackboneWrapper(nn.Module):
    """Wraps onnx2torch model to:
    1. Handle list/tuple outputs (pick the embedding tensor)
    2. Project from ONNX_EMBEDDING_DIM -> 192 if needed
    """
    def __init__(self, raw_model, onnx_dim, target_dim):
        super().__init__()
        self.raw_model = raw_model
        self.onnx_dim = onnx_dim
        self.needs_projection = (onnx_dim != target_dim)
        if self.needs_projection:
            self.projection = nn.Linear(onnx_dim, target_dim, bias=False)
            nn.init.xavier_uniform_(self.projection.weight)
            print(f'  Added projection layer: {onnx_dim} -> {target_dim}')
        else:
            print(f'  No projection needed (dims match: {onnx_dim})')

    def forward(self, x):
        out = self.raw_model(x)
        # onnx2torch may return list/tuple of tensors
        if isinstance(out, (list, tuple)):
            for t in out:
                if isinstance(t, torch.Tensor) and t.dim() == 2 and t.shape[1] == self.onnx_dim:
                    out = t
                    break
            else:
                out = out[-1] if isinstance(out[-1], torch.Tensor) else out[0]
        if self.needs_projection:
            out = self.projection(out)
        return out


pytorch_model = BackboneWrapper(
    pytorch_model_raw,
    onnx_dim=ONNX_EMBEDDING_DIM,
    target_dim=TARGET_EMBEDDING_DIM,
).to(DEVICE)
pytorch_model.eval()

# ============================================================
# Verify PyTorch model
# ============================================================
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 112, 112).to(DEVICE)
    dummy_output = pytorch_model(dummy_input)
    print(f'\nPyTorch model output shape: {dummy_output.shape}')
    assert dummy_output.shape == (1, TARGET_EMBEDDING_DIM), \
        f'Expected (1, {TARGET_EMBEDDING_DIM}), got {dummy_output.shape}'
    print(f'Model verified: (1,3,112,112) -> (1,{TARGET_EMBEDDING_DIM})')

# Cross-check raw model vs ONNX Runtime (before projection)
with torch.no_grad():
    raw_out = pytorch_model_raw(dummy_input)
    if isinstance(raw_out, (list, tuple)):
        for t in raw_out:
            if isinstance(t, torch.Tensor) and t.dim() == 2:
                raw_out = t
                break
    raw_out_np = raw_out.cpu().numpy()

cos_sim = np.dot(onnx_output.flatten(), raw_out_np.flatten()) / (
    np.linalg.norm(onnx_output) * np.linalg.norm(raw_out_np) + 1e-8)
print(f'ONNX Runtime vs PyTorch (raw) cosine similarity: {cos_sim:.6f} (should be > 0.99)')

## Cell 3 — Download & Prepare RFW African Dataset

In [ ]:
import zipfile
import shutil
from collections import defaultdict

# ============================================================
# Download RFW (Racial Faces in the Wild) dataset
# Website: http://www.whdeng.cn/RFW/testing.html
# ============================================================
RFW_DIR = '/content/rfw'
os.makedirs(RFW_DIR, exist_ok=True)

# RFW test data — contains African, Asian, Caucasian, Indian subsets
# Each subset has: images/ (identity folders) and pairs.txt (verification pairs)
RFW_URL = 'https://drive.google.com/uc?id=1kGxJOkNHsUMzhh_DLtoWOEGSdb-vOy9I'
RFW_ZIP = '/content/rfw_test.zip'

if not os.path.exists(os.path.join(RFW_DIR, 'African')):
    print('Downloading RFW dataset...')
    print('NOTE: If automatic download fails, manually download from:')
    print('http://www.whdeng.cn/RFW/testing.html')
    print('Then upload the zip to Colab and update the path below.\n')

    try:
        import gdown
        gdown.download(RFW_URL, RFW_ZIP, quiet=False)
    except Exception as e:
        print(f'Auto-download failed: {e}')
        print('\n=== MANUAL DOWNLOAD REQUIRED ===')
        print('1. Go to http://www.whdeng.cn/RFW/testing.html')
        print('2. Download the test dataset')
        print('3. Upload the zip file to Colab')
        print('4. Update RFW_ZIP path and re-run this cell')
        raise

    print('Extracting...')
    with zipfile.ZipFile(RFW_ZIP, 'r') as zf:
        zf.extractall(RFW_DIR)
    print('Extraction complete.')

# Locate African subset
# RFW structure: rfw/data/African/person_name/person_name_0001.jpg
# or: rfw/test/txts/African/African_pairs.txt
AFRICAN_IMG_DIR = None
AFRICAN_PAIRS_FILE = None

for root, dirs, files in os.walk(RFW_DIR):
    for d in dirs:
        if d == 'African':
            candidate = os.path.join(root, d)
            # Check if this has image subdirectories
            subdirs = [s for s in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, s))]
            if len(subdirs) > 10:  # Has many identity directories
                AFRICAN_IMG_DIR = candidate
    for f in files:
        if 'african' in f.lower() and 'pair' in f.lower():
            AFRICAN_PAIRS_FILE = os.path.join(root, f)

print(f'African images dir: {AFRICAN_IMG_DIR}')
print(f'African pairs file: {AFRICAN_PAIRS_FILE}')

# Count identities and images
identities = [d for d in os.listdir(AFRICAN_IMG_DIR)
              if os.path.isdir(os.path.join(AFRICAN_IMG_DIR, d))]
total_images = sum(
    len([f for f in os.listdir(os.path.join(AFRICAN_IMG_DIR, d)) if f.endswith(('.jpg', '.png'))])
    for d in identities
)
print(f'\nAfrican subset: {len(identities)} identities, {total_images} images')

# ============================================================
# Parse verification pairs
# Format: same-person pairs then different-person pairs
#   same:  name  idx1  idx2
#   diff:  name1 idx1  name2 idx2
# ============================================================
def parse_rfw_pairs(pairs_file, img_dir):
    """Parse RFW-format pairs file into (path1, path2, label) tuples."""
    pairs = []
    with open(pairs_file, 'r') as f:
        lines = f.readlines()

    # First line may contain the number of pairs per set
    idx = 0
    first_line = lines[0].strip().split()
    if len(first_line) <= 2 and first_line[0].isdigit():
        n_pairs = int(first_line[0])
        idx = 1
    else:
        n_pairs = None

    while idx < len(lines):
        line = lines[idx].strip()
        idx += 1
        if not line:
            continue

        parts = line.split('\t') if '\t' in line else line.split()

        if len(parts) == 3:
            # Same person: name idx1 idx2
            name, idx1, idx2 = parts[0], int(parts[1]), int(parts[2])
            path1 = _find_image(img_dir, name, idx1)
            path2 = _find_image(img_dir, name, idx2)
            if path1 and path2:
                pairs.append((path1, path2, 1))  # 1 = same person

        elif len(parts) == 4:
            # Different person: name1 idx1 name2 idx2
            name1, idx1 = parts[0], int(parts[1])
            name2, idx2 = parts[2], int(parts[3])
            path1 = _find_image(img_dir, name1, idx1)
            path2 = _find_image(img_dir, name2, idx2)
            if path1 and path2:
                pairs.append((path1, path2, 0))  # 0 = different person

    return pairs


def _find_image(img_dir, name, idx):
    """Find image file by identity name and index."""
    person_dir = os.path.join(img_dir, name)
    if not os.path.isdir(person_dir):
        return None

    # Try common naming patterns: name_0001.jpg, name_0001.png, 0001.jpg
    for ext in ['.jpg', '.png', '.jpeg']:
        for pattern in [f'{name}_{idx:04d}{ext}', f'{idx:04d}{ext}', f'{name}_{idx}{ext}']:
            path = os.path.join(person_dir, pattern)
            if os.path.exists(path):
                return path

    # Fallback: list directory and find by index
    files = sorted([f for f in os.listdir(person_dir) if f.endswith(('.jpg', '.png', '.jpeg'))])
    if 0 < idx <= len(files):
        return os.path.join(person_dir, files[idx - 1])

    return None


verification_pairs = parse_rfw_pairs(AFRICAN_PAIRS_FILE, AFRICAN_IMG_DIR)
same_pairs = [p for p in verification_pairs if p[2] == 1]
diff_pairs = [p for p in verification_pairs if p[2] == 0]
print(f'\nVerification pairs: {len(verification_pairs)} total')
print(f'  Same person: {len(same_pairs)}')
print(f'  Different person: {len(diff_pairs)}')

# ============================================================
# Create training Dataset with augmentations
# ============================================================
class RFWAfricanDataset(Dataset):
    """Dataset for fine-tuning: loads all identity images."""

    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.samples = []  # (image_path, label_idx)
        self.class_to_idx = {}

        identities = sorted([
            d for d in os.listdir(img_dir)
            if os.path.isdir(os.path.join(img_dir, d))
        ])

        for idx, identity in enumerate(identities):
            self.class_to_idx[identity] = idx
            identity_dir = os.path.join(img_dir, identity)
            for img_name in os.listdir(identity_dir):
                if img_name.endswith(('.jpg', '.png', '.jpeg')):
                    self.samples.append((
                        os.path.join(identity_dir, img_name),
                        idx
                    ))

        self.num_classes = len(identities)
        print(f'Dataset: {len(self.samples)} images, {self.num_classes} classes')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        img_path, label = self.samples[index]
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)
        else:
            img = transforms.ToTensor()(img)

        return img, label


# Training augmentations
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1,
        hue=0.05,
    ),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  # -> [-1, 1]
])

# Evaluation transform (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

# Create dataset and dataloader
train_dataset = RFWAfricanDataset(AFRICAN_IMG_DIR, transform=train_transform)

# Split into train/val (90/10)
from torch.utils.data import random_split
n_val = max(1, int(0.1 * len(train_dataset)))
n_train = len(train_dataset) - n_val
train_subset, val_subset = random_split(
    train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_subset, batch_size=64, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_subset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'\nTrain: {len(train_subset)} images')
print(f'Val: {len(val_subset)} images')
print(f'Batches per epoch: {len(train_loader)}')
print(f'Number of classes: {train_dataset.num_classes}')

## Cell 4 — Define ArcFace Loss + Training Loop

In [ ]:
import math
import torch.nn.functional as F


class ArcFaceLoss(nn.Module):
    """ArcFace (Additive Angular Margin) loss for face recognition.

    Adds an angular margin penalty to the target logit to enforce
    intra-class compactness and inter-class separability.

    Args:
        in_features: Embedding dimension (192 for MobileFaceNet)
        out_features: Number of classes (identities)
        s: Feature scale (default: 64.0)
        m: Angular margin in radians (default: 0.5)
    """

    def __init__(self, in_features, out_features, s=64.0, m=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)  # threshold
        self.mm = math.sin(math.pi - m) * m  # for numerical stability

        # Class weight matrix (learnable)
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        # Normalize embeddings and weights
        embeddings = F.normalize(embeddings, p=2, dim=1)
        weights = F.normalize(self.weight, p=2, dim=1)

        # Cosine similarity
        cosine = F.linear(embeddings, weights)
        sine = torch.sqrt(1.0 - torch.clamp(cosine * cosine, 0, 1))

        # cos(theta + m) = cos(theta)*cos(m) - sin(theta)*sin(m)
        phi = cosine * self.cos_m - sine * self.sin_m

        # Numerical stability: when cos(theta) < cos(pi - m), use fallback
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        # One-hot encode labels
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)

        # Apply margin only to the target class
        logits = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        logits *= self.s

        loss = F.cross_entropy(logits, labels)
        return loss


class FinetuneModel(nn.Module):
    """Wrapper that combines backbone + ArcFace head for training."""

    def __init__(self, backbone, embedding_dim=192, num_classes=100):
        super().__init__()
        self.backbone = backbone
        self.arcface = ArcFaceLoss(
            in_features=embedding_dim,
            out_features=num_classes,
            s=64.0,
            m=0.5,
        )

    def forward(self, images, labels=None):
        embeddings = self.backbone(images)
        if labels is not None:
            loss = self.arcface(embeddings, labels)
            return embeddings, loss
        return embeddings


# ============================================================
# Freeze strategy:
#   - Freeze early layers of the raw_model (CNN backbone)
#   - Keep projection layer (if present) fully trainable
#   - Keep ArcFace head fully trainable
# ============================================================
def freeze_early_layers(model, freeze_ratio=0.5):
    """Freeze the first `freeze_ratio` of the raw CNN backbone parameters."""
    # Only freeze inside the raw_model (the onnx2torch CNN), not projection
    raw_params = list(model.backbone.raw_model.parameters())
    n_freeze = int(len(raw_params) * freeze_ratio)

    for i, param in enumerate(raw_params):
        param.requires_grad = (i >= n_freeze)

    # Ensure projection layer is always trainable
    if model.backbone.needs_projection:
        for param in model.backbone.projection.parameters():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')
    print(f'Frozen: first {n_freeze} / {len(raw_params)} raw backbone param groups')
    if model.backbone.needs_projection:
        proj_params = sum(p.numel() for p in model.backbone.projection.parameters())
        print(f'Projection layer ({proj_params:,} params): always trainable')


# Build the fine-tuning model
model = FinetuneModel(
    backbone=pytorch_model,
    embedding_dim=TARGET_EMBEDDING_DIM,
    num_classes=train_dataset.num_classes,
).to(DEVICE)

freeze_early_layers(model, freeze_ratio=0.5)

# Optimizer: SGD with momentum (standard for face recognition fine-tuning)
optimizer = optim.SGD(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    momentum=0.9,
    weight_decay=5e-4,
)

# Learning rate schedule: cosine annealing
NUM_EPOCHS = 20
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

print(f'\nTraining config:')
print(f'  Epochs: {NUM_EPOCHS}')
print(f'  Batch size: 64')
print(f'  Optimizer: SGD (lr=1e-3, momentum=0.9, wd=5e-4)')
print(f'  Scheduler: CosineAnnealingLR')
print(f'  Loss: ArcFace (s=64, m=0.5)')
print(f'  Backbone embedding dim: {ONNX_EMBEDDING_DIM}')
print(f'  Output embedding dim: {TARGET_EMBEDDING_DIM}')

## Cell 5 — Fine-tuning Execution

In [ ]:
# ============================================================
# Training loop
# ============================================================
train_losses = []
val_losses = []
learning_rates = []
best_val_loss = float('inf')
best_epoch = 0

print('Starting fine-tuning...\n')

for epoch in range(NUM_EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for images, labels in pbar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        _, loss = model(images, labels)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})

    avg_train_loss = epoch_loss / max(n_batches, 1)
    train_losses.append(avg_train_loss)
    learning_rates.append(scheduler.get_last_lr()[0])

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            _, loss = model(images, labels)
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / max(val_batches, 1)
    val_losses.append(avg_val_loss)

    print(f'  Epoch {epoch+1}: train_loss={avg_train_loss:.4f}, '
          f'val_loss={avg_val_loss:.4f}, lr={scheduler.get_last_lr()[0]:.6f}')

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save({
            'epoch': epoch + 1,
            'backbone_state_dict': model.backbone.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val_loss,
            'train_loss': avg_train_loss,
        }, os.path.join(SAVE_DIR, 'best_mobilefacenet_african.pth'))
        print(f'  -> Saved best model (epoch {epoch+1}, val_loss={avg_val_loss:.4f})')

    scheduler.step()

print(f'\nTraining complete. Best model: epoch {best_epoch} (val_loss={best_val_loss:.4f})')

# ============================================================
# Plot training curves
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, NUM_EPOCHS+1), train_losses, 'b-', label='Train Loss')
ax1.plot(range(1, NUM_EPOCHS+1), val_losses, 'r-', label='Val Loss')
ax1.axvline(best_epoch, color='g', linestyle='--', label=f'Best (epoch {best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True)

ax2.plot(range(1, NUM_EPOCHS+1), learning_rates, 'g-')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

## Cell 6 — Evaluation on RFW African Pairs

In [ ]:
# ============================================================
# Load best checkpoint and evaluate on verification pairs
# ============================================================
best_ckpt = torch.load(
    os.path.join(SAVE_DIR, 'best_mobilefacenet_african.pth'),
    map_location=DEVICE,
)
model.backbone.load_state_dict(best_ckpt['backbone_state_dict'])
model.eval()
print(f'Loaded best model from epoch {best_ckpt["epoch"]} (val_loss={best_ckpt["val_loss"]:.4f})')


def extract_embedding(model, img_path, transform):
    """Extract a 192-dim embedding from an image."""
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        embedding = model.backbone(img_tensor)
        embedding = F.normalize(embedding, p=2, dim=1)
    return embedding.cpu().numpy().flatten()


# Compute embeddings and similarities for all verification pairs
print(f'\nEvaluating on {len(verification_pairs)} verification pairs...')
similarities = []
labels = []

for path1, path2, label in tqdm(verification_pairs, desc='Extracting embeddings'):
    try:
        emb1 = extract_embedding(model, path1, eval_transform)
        emb2 = extract_embedding(model, path2, eval_transform)
        sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
        similarities.append(sim)
        labels.append(label)
    except Exception as e:
        continue  # Skip corrupted images

similarities = np.array(similarities)
labels = np.array(labels)
print(f'Successfully evaluated {len(similarities)} pairs')

# ============================================================
# ROC Analysis
# ============================================================
fpr, tpr, thresholds = roc_curve(labels, similarities)
roc_auc = auc(fpr, tpr)

# Find optimal threshold (maximizes TPR - FPR, i.e., Youden's J statistic)
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

# Find threshold at FAR < 1%
far_1_idx = np.where(fpr <= 0.01)[0]
if len(far_1_idx) > 0:
    threshold_at_far1 = thresholds[far_1_idx[-1]]
    tpr_at_far1 = tpr[far_1_idx[-1]]
else:
    threshold_at_far1 = optimal_threshold
    tpr_at_far1 = 0.0

# Compute metrics at optimal threshold
predictions = (similarities >= optimal_threshold).astype(int)
accuracy = np.mean(predictions == labels)
tp = np.sum((predictions == 1) & (labels == 1))
fp = np.sum((predictions == 1) & (labels == 0))
tn = np.sum((predictions == 0) & (labels == 0))
fn = np.sum((predictions == 0) & (labels == 1))

far = fp / max(fp + tn, 1)  # False Accept Rate
frr = fn / max(fn + tp, 1)  # False Reject Rate

print(f'\n{"="*60}')
print(f' EVALUATION RESULTS — RFW African Verification Pairs')
print(f'{"="*60}')
print(f' AUC:                  {roc_auc:.4f}')
print(f' Optimal threshold:    {optimal_threshold:.4f}')
print(f' Accuracy:             {accuracy*100:.2f}%')
print(f' FAR:                  {far*100:.2f}%')
print(f' FRR:                  {frr*100:.2f}%')
print(f'{"="*60}')
print(f' At FAR < 1%:')
print(f'   Threshold:          {threshold_at_far1:.4f}')
print(f'   TPR (recall):       {tpr_at_far1*100:.2f}%')
print(f'{"="*60}')

# Recommended threshold for the Flutter app
# Use the FAR<1% threshold for security-sensitive field deployment
RECOMMENDED_THRESHOLD = round(float(threshold_at_far1), 2)
print(f'\n>>> RECOMMENDED THRESHOLD for Flutter app: {RECOMMENDED_THRESHOLD}')
print(f'>>> Update defaultThreshold in distance_metrics.dart to: {RECOMMENDED_THRESHOLD}')

# ============================================================
# Plot ROC curve and similarity distributions
# ============================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# ROC curve
ax1.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.5)
ax1.scatter(fpr[optimal_idx], tpr[optimal_idx], c='red', s=100, zorder=5,
            label=f'Optimal (t={optimal_threshold:.3f})')
ax1.set_xlabel('False Positive Rate (FAR)')
ax1.set_ylabel('True Positive Rate (1-FRR)')
ax1.set_title('ROC Curve — African Face Verification')
ax1.legend(loc='lower right')
ax1.grid(True)

# Similarity distributions
same_sims = similarities[labels == 1]
diff_sims = similarities[labels == 0]
ax2.hist(same_sims, bins=50, alpha=0.7, color='green', label='Same person', density=True)
ax2.hist(diff_sims, bins=50, alpha=0.7, color='red', label='Different person', density=True)
ax2.axvline(optimal_threshold, color='blue', linestyle='--', linewidth=2,
            label=f'Threshold = {optimal_threshold:.3f}')
ax2.set_xlabel('Cosine Similarity')
ax2.set_ylabel('Density')
ax2.set_title('Similarity Distribution — African Faces')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'evaluation_results.png'), dpi=150)
plt.show()

## Cell 7 — Export to TFLite

In [ ]:
import tensorflow as tf

# ============================================================
# Step 1: Export PyTorch backbone to ONNX
# ============================================================
print('Step 1: PyTorch -> ONNX')
model.backbone.eval()

# PyTorch uses NCHW format
dummy_input = torch.randn(1, 3, 112, 112).to(DEVICE)
ONNX_EXPORT_PATH = '/content/mobilefacenet_african_v2.onnx'

# Export the wrapper (which handles list->tensor internally) via tracing
# torch.onnx.export uses tracing, so the wrapper's forward runs and
# the ONNX graph captures a single tensor output.
torch.onnx.export(
    model.backbone,
    dummy_input,
    ONNX_EXPORT_PATH,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'embedding': {0: 'batch_size'},
    },
    opset_version=12,
    do_constant_folding=True,
)

# Verify ONNX export
onnx_model = onnx.load(ONNX_EXPORT_PATH)
onnx.checker.check_model(onnx_model)
print(f'  ONNX model exported: {os.path.getsize(ONNX_EXPORT_PATH)/1024/1024:.1f} MB')

# Verify output matches
with torch.no_grad():
    pytorch_out = model.backbone(dummy_input).cpu().numpy()

sess = ort.InferenceSession(ONNX_EXPORT_PATH, providers=['CPUExecutionProvider'])
onnx_out = sess.run(None, {'input': dummy_input.cpu().numpy()})[0]
cos_sim = np.dot(pytorch_out.flatten(), onnx_out.flatten()) / (
    np.linalg.norm(pytorch_out) * np.linalg.norm(onnx_out))
print(f'  PyTorch vs ONNX cosine similarity: {cos_sim:.6f}')

# ============================================================
# Step 2: ONNX -> TensorFlow SavedModel
# ============================================================
print('\nStep 2: ONNX -> TensorFlow SavedModel')
from onnx_tf.backend import prepare

onnx_model = onnx.load(ONNX_EXPORT_PATH)
tf_rep = prepare(onnx_model)

TF_SAVED_MODEL_DIR = '/content/mobilefacenet_african_v2_saved_model'
tf_rep.export_graph(TF_SAVED_MODEL_DIR)
print(f'  SavedModel exported to: {TF_SAVED_MODEL_DIR}')

# ============================================================
# Step 3: TensorFlow SavedModel -> TFLite
# ============================================================
print('\nStep 3: TF SavedModel -> TFLite')

# The ONNX model expects NCHW input (1, 3, 112, 112) from PyTorch.
# Flutter app sends NHWC input (1, 112, 112, 3).
# We wrap the model to handle the transpose.

class NHWCWrapper(tf.Module):
    """Wraps the NCHW model to accept NHWC input for TFLite/Flutter."""

    def __init__(self, saved_model_dir):
        super().__init__()
        self.model = tf.saved_model.load(saved_model_dir)
        # Find the concrete function
        self.infer = self.model.signatures.get(
            'serving_default',
            list(self.model.signatures.values())[0] if self.model.signatures else None
        )

    @tf.function(input_signature=[tf.TensorSpec(shape=[1, 112, 112, 3], dtype=tf.float32)])
    def __call__(self, nhwc_input):
        # Transpose NHWC -> NCHW for the model
        nchw_input = tf.transpose(nhwc_input, perm=[0, 3, 1, 2])
        # Run inference
        result = self.infer(input=nchw_input)
        # Get the output tensor (key name varies)
        output_key = list(result.keys())[0]
        embedding = result[output_key]
        # L2 normalize
        embedding = tf.math.l2_normalize(embedding, axis=1)
        return embedding


try:
    # Try the wrapper approach first
    wrapper = NHWCWrapper(TF_SAVED_MODEL_DIR)

    # Test the wrapper
    test_nhwc = tf.random.normal([1, 112, 112, 3])
    test_out = wrapper(test_nhwc)
    print(f'  Wrapper output shape: {test_out.shape}')
    assert test_out.shape == (1, 192), f'Expected (1, 192), got {test_out.shape}'

    # Convert to TFLite using the wrapper
    concrete_func = wrapper.__call__.get_concrete_function(
        tf.TensorSpec(shape=[1, 112, 112, 3], dtype=tf.float32)
    )

    converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func], wrapper)

except Exception as e:
    print(f'  Wrapper approach failed ({e}), using direct conversion...')
    # Fallback: direct conversion without wrapper
    # The Flutter code will need to handle transpose
    converter = tf.lite.TFLiteConverter.from_saved_model(TF_SAVED_MODEL_DIR)

converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]  # float16 quantization

tflite_model = converter.convert()

TFLITE_PATH = '/content/mobilefacenet_african_v2.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f'\n  TFLite model saved: {TFLITE_PATH}')
print(f'  Size: {tflite_size_mb:.2f} MB')

# If model is too large, apply stronger quantization
if tflite_size_mb > 5.0:
    print(f'\n  Model exceeds 5MB ({tflite_size_mb:.1f}MB), applying full float16 quantization...')
    converter = tf.lite.TFLiteConverter.from_saved_model(TF_SAVED_MODEL_DIR)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    with open(TFLITE_PATH, 'wb') as f:
        f.write(tflite_model)
    tflite_size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
    print(f'  Final size: {tflite_size_mb:.2f} MB')

# ============================================================
# Validate TFLite model
# ============================================================
print('\nValidating TFLite model...')
interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'  Input:  shape={input_details[0]["shape"]}, dtype={input_details[0]["dtype"]}')
print(f'  Output: shape={output_details[0]["shape"]}, dtype={output_details[0]["dtype"]}')

# Validate with a test input
test_input = np.random.randn(1, 112, 112, 3).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]['index'])

print(f'  Test output shape: {tflite_output.shape}')
print(f'  Output range: [{tflite_output.min():.4f}, {tflite_output.max():.4f}]')
print(f'  L2 norm: {np.linalg.norm(tflite_output):.4f} (should be ~1.0)')

# Compare with PyTorch output for same input
# Transpose NHWC -> NCHW for PyTorch
test_input_nchw = np.transpose(test_input, (0, 3, 1, 2))
with torch.no_grad():
    pt_out = model.backbone(torch.from_numpy(test_input_nchw).to(DEVICE))
    pt_out = F.normalize(pt_out, p=2, dim=1).cpu().numpy()

cos_sim = np.dot(tflite_output.flatten(), pt_out.flatten()) / (
    np.linalg.norm(tflite_output) * np.linalg.norm(pt_out))
print(f'\n  TFLite vs PyTorch cosine similarity: {cos_sim:.6f}')
print(f'  (should be > 0.99 — if lower, check float16 quantization impact)')

assert input_details[0]['shape'].tolist() == [1, 112, 112, 3], \
    f'Input shape mismatch: {input_details[0]["shape"]}'
assert output_details[0]['shape'].tolist() == [1, 192], \
    f'Output shape mismatch: {output_details[0]["shape"]}'

print('\nTFLite model validation PASSED')

## Cell 8 — Download & Deploy

In [ ]:
import shutil

# ============================================================
# Copy to Google Drive for persistence
# ============================================================
drive_tflite = os.path.join(SAVE_DIR, 'mobilefacenet_african_v2.tflite')
shutil.copy(TFLITE_PATH, drive_tflite)
print(f'Model saved to Google Drive: {drive_tflite}')

# Also save as mobilefacenet.tflite (drop-in replacement)
drive_dropin = os.path.join(SAVE_DIR, 'mobilefacenet.tflite')
shutil.copy(TFLITE_PATH, drive_dropin)
print(f'Drop-in replacement: {drive_dropin}')

# ============================================================
# Download in Colab
# ============================================================
try:
    from google.colab import files
    print('\nDownloading TFLite model...')
    files.download(TFLITE_PATH)
except ImportError:
    print(f'\nNot in Colab. Model is at: {TFLITE_PATH}')

# ============================================================
# Deployment instructions
# ============================================================
print(f'''
{'='*60}
 DEPLOYMENT INSTRUCTIONS
{'='*60}

1. Copy the model file:
   cp mobilefacenet_african_v2.tflite \\
     packages/digit_face_verification/assets/models/mobilefacenet.tflite

2. Update distance_metrics.dart:
   static const double defaultThreshold = {RECOMMENDED_THRESHOLD};

3. Update face_embedding_model.dart:
   this.modelVersion = 'mobilefacenet_african_v2'

4. Update face_embedding_repository.dart:
   String modelVersion = 'mobilefacenet_african_v2'

5. IMPORTANT: Re-register all faces after model update.
   Embeddings from v1 are NOT compatible with v2.

{'='*60}
 MODEL PERFORMANCE SUMMARY
{'='*60}
 Accuracy:           {accuracy*100:.2f}%
 AUC:                {roc_auc:.4f}
 FAR:                {far*100:.2f}%
 FRR:                {frr*100:.2f}%
 Optimal threshold:  {optimal_threshold:.4f}
 Recommended (app):  {RECOMMENDED_THRESHOLD}
 Model size:         {tflite_size_mb:.2f} MB
 Model version:      mobilefacenet_african_v2
{'='*60}
''')